<!-- MUTERBANDUNG_NOTEBOOK_STATUS_V1 -->
# Notebook Status - Pendukung Sentiment

| Item | Isi |
|---|---|
| Status | PENDUKUNG_SENTIMENT |
| Fungsi | Persiapan dan penggabungan data review untuk pipeline sentiment wisata. |
| Input utama | Review mentah dan master review wisata. |
| Output utama | Dataset review gabungan/bersih untuk proses sentiment. |
| Aturan pakai | Jangan dijalankan sebelum path input dan file review dikunci ulang. |
| Catatan | Bukan notebook utama. Keputusan akhir tetap dicatat di `wisata_training.ipynb`. |

---


# Tahap 1: Penggabungan Data Ulasan (Review Aggregation)

**Tujuan:**
Mengumpulkan semua file CSV ulasan yang terpisah per wilayah (Kota Bandung, Kabupaten Bandung, dll) menjadi satu dataset master ulasan (Master Reviews). Data ini nantinya akan menjadi fondasi untuk ekstraksi skor sentimen (NLP) agar mesin rekomendasi tidak hanya melihat jarak dan harga, tapi juga kepuasan pengunjung sesungguhnya.

In [ ]:
import pandas as pd
import os
import glob

# Tentukan folder tempat data review berada
dataset_folder = 'Dataset'

# Kita hanya mengambil file yang sudah dibersihkan sebelumnya 
# (mengandung kata 'dataset_cleaned_')
file_pattern = os.path.join(dataset_folder, 'dataset_cleaned_*.csv')
review_files = glob.glob(file_pattern)

print("File review yang ditemukan:")
for f in review_files:
    print(f"- {f}")

### Membaca dan Menggabungkan CSV
Proses ini menggunakan perulangan standar untuk membaca tiap file, menambahkannya ke dalam list, dan kemudian menggabungkannya dengan `pd.concat`.

In [ ]:
dataframes = []

for file_path in review_files:
    try:
        # Baca file CSV
        df_temp = pd.read_csv(file_path)
        
        # Kita tambahkan kolom asal file buat jaga-jaga kalau butuh melacak data
        df_temp['source_file'] = os.path.basename(file_path)
        
        dataframes.append(df_temp)
        print(f"Berhasil memuat {len(df_temp)} baris dari {os.path.basename(file_path)}")
    except Exception as e:
        print(f"Gagal membaca {file_path}. Error: {e}")

# Gabungkan semua dataframe yang ada di dalam list
if len(dataframes) > 0:
    df_all_reviews = pd.concat(dataframes, ignore_index=True)
    print(f"\nTotal keseluruhan data ulasan yang digabung: {len(df_all_reviews)} baris.")
else:
    print("\nTidak ada data yang berhasil digabung.")

### Standarisasi dan Pembersihan Awal
Terkadang data *scraping* memiliki nama kolom yang berbeda atau ulasan yang kosong (misal pengguna hanya menekan Bintang tanpa menulis ulasan). Untuk keperluan Analisis Sentimen (NLP), kita wajib membuang baris yang tidak memiliki teks ulasan.

In [ ]:
print("Struktur kolom sebelum standarisasi:")
print(df_all_reviews.columns.tolist())

# Opsional: Ubah nama kolom agar lebih seragam dan mudah dibaca (huruf kecil semua)
df_all_reviews.columns = [str(col).lower().replace(' ', '_') for col in df_all_reviews.columns]

# (Asumsi: Teks ulasan ada di kolom ke-5 atau bernama 'text' / 'review')
# Anda perlu menyesuaikan nama kolom teks ulasannya sesuai dengan struktur asli CSV Anda.
# Misalnya nama kolom aslinya adalah 'text' atau 'review_text'.
kolom_teks = df_all_reviews.columns[4] # Seringkali hasil scraping teks ada di indeks ini

print(f"\nKolom yang diasumsikan sebagai teks ulasan: '{kolom_teks}'")

# Menghapus baris yang teks ulasannya kosong (NaN)
jumlah_awal = len(df_all_reviews)
df_all_reviews = df_all_reviews.dropna(subset=[kolom_teks])
jumlah_setelah = len(df_all_reviews)

print(f"\nBaris tanpa ulasan teks dihapus: {jumlah_awal - jumlah_setelah} baris.")
print(f"Sisa data ulasan yang siap masuk mesin NLP: {jumlah_setelah} baris.")

display(df_all_reviews.head(3))

### Menyimpan Hasil Penggabungan
Data master ulasan ini disimpan, dan nantinya akan digunakan di *notebook* terpisah khusus untuk proses NLP (IndoBERT/VADER) agar kodenya tidak terlalu panjang di satu tempat.

In [ ]:
output_path = 'Dataset/master_reviews_gabungan.csv'
df_all_reviews.to_csv(output_path, index=False)
print(f"Berhasil menyimpan data gabungan ke: {output_path}")